In [2]:
# Cell 1 - Setup. No extra installs needed - just torch, already available.
from google.colab import drive
drive.mount('/content/drive')
import os
os.chdir('/content/drive/MyDrive/ML_final')

import torch
print('torch:', torch.__version__, '| CUDA:', torch.cuda.is_available())

import pandas as pd
import numpy as np

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
torch: 2.11.0+cpu | CUDA: False


# Cell 2 - Load processed data (identical source as DLinear notebook)


In [3]:
train = pd.read_pickle('data/train_processed.pkl')
train = train.sort_values(['Store','Dept','Date'])
print(train['Date'].min(), train['Date'].max())

2010-02-05 00:00:00 2012-10-26 00:00:00


# Cell 3 - WMAE metric (same scoring function used everywhere)


In [4]:
def wmae(y_true, y_pred, is_holiday):
    weights = np.where(is_holiday, 5, 1)
    return np.sum(weights * np.abs(y_true - y_pred)) / np.sum(weights)

# Cell 4 - Select series and build clean per-series arrays.


In [5]:
# PatchTST here uses RevIN (instance normalization inside the model), so
# unlike DLinear we don't need a separate global scaler.
series_totals = (
    train.groupby(['Store','Dept'])['Weekly_Sales']
    .sum()
    .sort_values(ascending=False)
)
top_series = series_totals.index.tolist()

INPUT_LEN = 52
OUTPUT_LEN = 13
VAL_LEN = 39          # same holdout horizon as DLinear (3 x 13-week blocks)
MIN_LEN = INPUT_LEN + VAL_LEN + 10

series_data = {}   # (store,dept) -> dict with 'values' and 'is_holiday'
for store, dept in top_series:
    sub = train[(train['Store']==store) & (train['Dept']==dept)].sort_values('Date')
    if len(sub) < MIN_LEN:
        continue
    sub = sub.set_index('Date').asfreq('W-FRI')
    sub['Weekly_Sales'] = sub['Weekly_Sales'].interpolate().bfill().ffill()
    sub['IsHoliday'] = sub['IsHoliday'].astype(float).fillna(0.0).astype(bool)
    series_data[(store, dept)] = {
        'values': sub['Weekly_Sales'].values.astype(np.float32),
        'is_holiday': sub['IsHoliday'].values,
        'dates': sub.index
    }

keys_kept = list(series_data.keys())
print('Series successfully built:', len(keys_kept))

Series successfully built: 2873


# Cell 5 - Split each series into fit/val, then build (input, output) windows


In [6]:
# from the fit portion only. A small tail of fit windows is held out as a
# monitor set for early stopping (same purpose as DLinear's val_series).
fit_raw, val_raw = {}, {}
for key, d in series_data.items():
    vals = d['values']
    fit_raw[key] = vals[:-VAL_LEN]
    val_raw[key] = vals[-VAL_LEN:]

X_train, Y_train, X_monitor, Y_monitor = [], [], [], []
for key, fseries in fit_raw.items():
    n = len(fseries)
    max_start = n - INPUT_LEN - OUTPUT_LEN
    if max_start < 1:
        continue
    # hold out the last window of each series as the monitor sample
    for start in range(max_start):  # 0 .. max_start-1 -> training
        X_train.append(fseries[start:start+INPUT_LEN])
        Y_train.append(fseries[start+INPUT_LEN:start+INPUT_LEN+OUTPUT_LEN])
    X_monitor.append(fseries[max_start:max_start+INPUT_LEN])
    Y_monitor.append(fseries[max_start+INPUT_LEN:max_start+INPUT_LEN+OUTPUT_LEN])

X_train = np.stack(X_train).astype(np.float32)
Y_train = np.stack(Y_train).astype(np.float32)
X_monitor = np.stack(X_monitor).astype(np.float32)
Y_monitor = np.stack(Y_monitor).astype(np.float32)

print('Train windows:', X_train.shape, '| Monitor windows:', X_monitor.shape)

Train windows: (111752, 52) | Monitor windows: (2873, 52)


# Cell 6 - PatchTST implementation in plain PyTorch.


In [7]:
# Core ingredients from the paper: (1) instance normalization (RevIN),
# (2) splitting the lookback window into patches instead of feeding raw
# timesteps, (3) a plain transformer encoder over the patch tokens,
# (4) a flatten + linear head projecting to the forecast horizon.
import torch.nn as nn

PATCH_LEN = 13
STRIDE = 13
NUM_PATCHES = INPUT_LEN // PATCH_LEN  # 52 / 13 = 4, no overlap - clean and simple

class RevIN(nn.Module):
    """Instance normalization: subtract/divide by each window's own
    mean/std before the model, then undo it on the output."""
    def __init__(self, eps=1e-5):
        super().__init__()
        self.eps = eps

    def forward(self, x):
        # x: (batch, input_len)
        mean = x.mean(dim=1, keepdim=True)
        std = x.std(dim=1, keepdim=True) + self.eps
        x_norm = (x - mean) / std
        return x_norm, mean, std

class PatchTST(nn.Module):
    def __init__(self, input_len, output_len, patch_len, num_patches,
                 d_model=64, nhead=4, num_layers=3, dim_feedforward=256, dropout=0.1):
        super().__init__()
        self.revin = RevIN()
        self.patch_len = patch_len
        self.num_patches = num_patches

        self.patch_embed = nn.Linear(patch_len, d_model)
        self.pos_embed = nn.Parameter(torch.randn(1, num_patches, d_model) * 0.02)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True, activation='gelu'
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.head = nn.Linear(num_patches * d_model, output_len)

    def forward(self, x):
        # x: (batch, input_len)
        x_norm, mean, std = self.revin(x)
        patches = x_norm.reshape(x.size(0), self.num_patches, self.patch_len)
        tokens = self.patch_embed(patches) + self.pos_embed
        encoded = self.encoder(tokens)
        flat = encoded.reshape(x.size(0), -1)
        out_norm = self.head(flat)
        out = out_norm * std + mean  # undo RevIN
        return out

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model_patchtst = PatchTST(INPUT_LEN, OUTPUT_LEN, PATCH_LEN, NUM_PATCHES).to(device)
print(model_patchtst)

PatchTST(
  (revin): RevIN()
  (patch_embed): Linear(in_features=13, out_features=64, bias=True)
  (encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-2): 3 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
        )
        (linear1): Linear(in_features=64, out_features=256, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=256, out_features=64, bias=True)
        (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (head): Linear(in_features=256, out_features=13, bias=True)
)


# Cell 7 - Train with early stopping on the monitor set, same spirit as


In [8]:
# DLinear's val_series-based EarlyStopping callback.
from torch.utils.data import TensorDataset, DataLoader

X_train_t = torch.from_numpy(X_train)
Y_train_t = torch.from_numpy(Y_train)
X_monitor_t = torch.from_numpy(X_monitor).to(device)
Y_monitor_t = torch.from_numpy(Y_monitor).to(device)

train_loader = DataLoader(TensorDataset(X_train_t, Y_train_t), batch_size=1024, shuffle=True)

loss_fn = torch.nn.HuberLoss(delta=1.0)
optimizer = torch.optim.AdamW(model_patchtst.parameters(), lr=5e-4, weight_decay=1e-2)
N_EPOCHS = 30
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=N_EPOCHS)

best_val_loss = float('inf')
patience, patience_counter = 5, 0
best_state = None

for epoch in range(N_EPOCHS):
    model_patchtst.train()
    train_loss_sum, n_batches = 0.0, 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        pred = model_patchtst(xb)
        loss = loss_fn(pred, yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_patchtst.parameters(), 0.5)
        optimizer.step()
        train_loss_sum += loss.item()
        n_batches += 1
    scheduler.step()

    model_patchtst.eval()
    with torch.no_grad():
        val_pred = model_patchtst(X_monitor_t)
        val_loss = loss_fn(val_pred, Y_monitor_t).item()

    print(f"Epoch {epoch+1} - train_loss: {train_loss_sum/n_batches:.5f} - val_loss: {val_loss:.5f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = {k: v.clone() for k, v in model_patchtst.state_dict().items()}
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print("Early stopping.")
            break

model_patchtst.load_state_dict(best_state)

Epoch 1 - train_loss: 1997.57204 - val_loss: 3036.56152
Epoch 2 - train_loss: 1702.03018 - val_loss: 2767.45435
Epoch 3 - train_loss: 1623.07162 - val_loss: 2661.41113
Epoch 4 - train_loss: 1578.00157 - val_loss: 2580.60181
Epoch 5 - train_loss: 1550.03962 - val_loss: 2549.91162
Epoch 6 - train_loss: 1526.92854 - val_loss: 2548.68652
Epoch 7 - train_loss: 1508.64843 - val_loss: 2572.02441
Epoch 8 - train_loss: 1495.01848 - val_loss: 2516.44629
Epoch 9 - train_loss: 1482.04080 - val_loss: 2527.88379
Epoch 10 - train_loss: 1469.24936 - val_loss: 2476.57300
Epoch 11 - train_loss: 1459.21567 - val_loss: 2497.58594
Epoch 12 - train_loss: 1449.71024 - val_loss: 2506.25781
Epoch 13 - train_loss: 1443.12108 - val_loss: 2539.72046
Epoch 14 - train_loss: 1434.90252 - val_loss: 2516.92480
Epoch 15 - train_loss: 1429.52625 - val_loss: 2462.48145
Epoch 16 - train_loss: 1419.52910 - val_loss: 2478.65625
Epoch 17 - train_loss: 1416.31585 - val_loss: 2453.68530
Epoch 18 - train_loss: 1412.17695 - val_

<All keys matched successfully>

# Cell 8 - Predict 39 weeks ahead per series by rolling the 13-week output


In [9]:
# forward 3 times (39 / 13 = 3), feeding predictions back in as context -
# same idea as darts automatically doing when n > output_chunk_length.
model_patchtst.eval()
all_true, all_pred, all_holiday = [], [], []

with torch.no_grad():
    for key in keys_kept:
        context = fit_raw[key][-INPUT_LEN:].copy()
        true_vals = val_raw[key]
        preds = []
        for _ in range(VAL_LEN // OUTPUT_LEN):
            x = torch.from_numpy(context[-INPUT_LEN:]).unsqueeze(0).to(device)
            block = model_patchtst(x).cpu().numpy().flatten()
            preds.extend(block)
            context = np.concatenate([context, block])

        preds = np.array(preds[:VAL_LEN])
        all_true.extend(true_vals)
        all_pred.extend(preds)
        all_holiday.extend(series_data[key]['is_holiday'][-VAL_LEN:])

all_true = np.array(all_true)
all_pred = np.array(all_pred)
all_holiday = np.array(all_holiday)
print('Done. Total validation points:', len(all_true))

Done. Total validation points: 112047


# Cell 9 - Evaluate


In [10]:
patchtst_wmae = wmae(all_true, all_pred, all_holiday)
print('PatchTST (custom PyTorch) WMAE:', patchtst_wmae)
print('Holiday WMAE:    ', wmae(all_true[all_holiday], all_pred[all_holiday], all_holiday[all_holiday]))
print('Non-holiday WMAE:', wmae(all_true[~all_holiday], all_pred[~all_holiday], all_holiday[~all_holiday]))

PatchTST (custom PyTorch) WMAE: 1719.3632017914535
Holiday WMAE:     1834.0180393795151
Non-holiday WMAE: 1688.5345303411184


# Cell 10 - Save model locally (same pattern as LightGBM/XGBoost notebooks)


In [12]:
import os
import joblib

os.makedirs('models', exist_ok=True)
joblib.dump(model_patchtst.cpu(), 'models/patchtst_best.pkl')
print('Saved.')

Saved.
